# Vessel-Cargo Optimization (Cargill Hackathon)

This notebook implements a CP-SAT optimization model to assign vessels to cargoes, maximizing profit while considering:
- Bunker forward curves for accurate fuel pricing
- Market freight rates for opportunity cost analysis
- Capacity, laycan, and routing constraints

In [1]:
# Cell 1: Load all CSVs
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import re

base = Path("data")

# Load vessel data
print("Loading vessel data...")
cargill_vessels = pd.read_csv(base / "Cargill_Capesize_Vessels.csv")
market_vessels = pd.read_csv(base / "Market_Vessels_Formatted.csv")

# Load cargo data
print("Loading cargo data...")
cargill_cargoes = pd.read_csv(base / "Cargill_Committed_Cargoes_Structured.csv")
market_cargoes = pd.read_csv(base / "Market_Cargoes_Structured.csv")

# Load port distances
print("Loading port distances...")
port_distances = pd.read_csv(base / "Port Distances.csv")

# Load port locations
print("Loading port locations...")
port_locations = pd.read_csv(base / "port_locations.csv")

# Load bunker forward curve
print("Loading bunker forward curve...")
bunker_forward_curve = pd.read_csv(base / "bunker_forward_curve.csv")

# Load freight rates
print("Loading freight rates...")
freight_rates = pd.read_csv(base / "freight_rates.csv")

print("\n✓ All data loaded successfully!")
print(f"Cargill vessels: {len(cargill_vessels)}")
print(f"Market vessels: {len(market_vessels)}")
print(f"Cargill cargoes: {len(cargill_cargoes)}")
print(f"Market cargoes: {len(market_cargoes)}")
print(f"Port distances: {len(port_distances)}")
print(f"Bunker locations: {len(bunker_forward_curve)}")
print(f"Freight routes: {len(freight_rates)}")

Loading vessel data...
Loading cargo data...
Loading port distances...
Loading port locations...
Loading bunker forward curve...
Loading freight rates...

✓ All data loaded successfully!
Cargill vessels: 4
Market vessels: 11
Cargill cargoes: 3
Market cargoes: 8
Port distances: 15533
Bunker locations: 18
Freight routes: 4


In [5]:
# Cell 2: Preprocess and normalize data

def normalize_port_name(port_str):
    """Convert 'Port, Country' to 'PORT_COUNTRY' format for distance lookup"""
    if pd.isna(port_str):
        return None
    text = str(port_str).strip().strip('"').strip()
    # Remove prefixes like "Discharging", "Loading", etc.
    text = re.sub(r"^(Discharging|Loading|At anchor|Anchored|At|In)\s+", "", text, flags=re.IGNORECASE).strip()
    # Extract "Port, Country" -> "PORT_COUNTRY"
    m = re.match(r"^([^,]+),\s*(.+)$", text)
    if m:
        port = m.group(1).strip().upper().replace(" ", "_")
        country = m.group(2).strip().upper().replace(" ", "_")
        return f"{port}_{country}"
    # If no comma, try to match as-is (uppercase, spaces to underscores)
    return text.upper().replace(" ", "_")

# Preprocess vessels
print("Preprocessing vessels...")

# Cargill vessels
cargill_vessels_processed = cargill_vessels.copy()
cargill_vessels_processed['vessel_name'] = cargill_vessels_processed['Vessel Name']
cargill_vessels_processed['deadweight_tonnage_dwt'] = cargill_vessels_processed['DWT (MT)']
cargill_vessels_processed['hire_rate_usd_per_day'] = cargill_vessels_processed['Hire Rate (USD/day)']
cargill_vessels_processed['economical_speed_knots'] = cargill_vessels_processed['Economical Speed Ballast (kn)']  # Use ballast for ballast leg
cargill_vessels_processed['sea_consumption_mt_per_day'] = cargill_vessels_processed['Economical Sea Consumption Ballast (mt/day)']
cargill_vessels_processed['port_consumption_mt_per_day'] = cargill_vessels_processed['Port Consumption Working (mt/day)']
cargill_vessels_processed['current_position_port'] = cargill_vessels_processed['Current Position / Status'].apply(normalize_port_name)
cargill_vessels_processed['estimated_time_of_departure'] = pd.to_datetime(cargill_vessels_processed['ETD'])
cargill_vessels_processed['speed_laden'] = cargill_vessels_processed['Economical Speed Laden (kn)']
cargill_vessels_processed['consumption_laden'] = cargill_vessels_processed['Economical Sea Consumption Laden (mt/day)']

# Market vessels (no hire rate column, set to 0 or use market rate)
market_vessels_processed = market_vessels.copy()
market_vessels_processed['vessel_name'] = market_vessels_processed['Vessel Name']
market_vessels_processed['deadweight_tonnage_dwt'] = market_vessels_processed['DWT (MT)']
market_vessels_processed['hire_rate_usd_per_day'] = 0  # Market vessels - will use freight rate instead
market_vessels_processed['economical_speed_knots'] = market_vessels_processed['Economical Speed Ballast (kn)']
market_vessels_processed['sea_consumption_mt_per_day'] = market_vessels_processed['Economical Sea Consumption Ballast (mt/day)']
market_vessels_processed['port_consumption_mt_per_day'] = market_vessels_processed['Port Consumption Working (mt/day)']
market_vessels_processed['current_position_port'] = market_vessels_processed['Current Position / Status'].apply(normalize_port_name)
market_vessels_processed['estimated_time_of_departure'] = pd.to_datetime(market_vessels_processed['ETD'])
market_vessels_processed['speed_laden'] = market_vessels_processed['Economical Speed Laden (kn)']
market_vessels_processed['consumption_laden'] = market_vessels_processed['Economical Sea Consumption Laden (mt/day)']

# Preprocess cargoes
print("Preprocessing cargoes...")

# Cargill cargoes
cargill_cargoes_processed = cargill_cargoes.copy()
cargill_cargoes_processed['cargo_id'] = ['CARGILL_' + str(i+1) for i in range(len(cargill_cargoes_processed))]
cargill_cargoes_processed['quantity_mt'] = cargill_cargoes_processed['Quantity_MT']
cargill_cargoes_processed['freight_rate_usd_per_mt'] = cargill_cargoes_processed['Freight_Rate_USD_PMT']
cargill_cargoes_processed['commission_percent'] = cargill_cargoes_processed['Commission_Percent'] / 100.0
cargill_cargoes_processed['laycan_start_date'] = pd.to_datetime(cargill_cargoes_processed['Laycan_Start'])
cargill_cargoes_processed['laycan_end_date'] = pd.to_datetime(cargill_cargoes_processed['Laycan_End'])
cargill_cargoes_processed['load_port'] = cargill_cargoes_processed['Load_Port_Primary'].apply(normalize_port_name)
cargill_cargoes_processed['discharge_port'] = cargill_cargoes_processed['Discharge_Port_Primary'].apply(normalize_port_name)
cargill_cargoes_processed['port_costs_usd'] = cargill_cargoes_processed['Port_Cost_USD'].fillna(0)
cargill_cargoes_processed['load_turn_time_hours'] = cargill_cargoes_processed['Load_Turn_Time_Hours'].fillna(12)
cargill_cargoes_processed['discharge_turn_time_hours'] = cargill_cargoes_processed['Discharge_Turn_Time_Hours'].fillna(24)
cargill_cargoes_processed['route'] = cargill_cargoes_processed['Route']

# Market cargoes
market_cargoes_processed = market_cargoes.copy()
market_cargoes_processed['cargo_id'] = ['MARKET_' + str(i+1) for i in range(len(market_cargoes_processed))]
market_cargoes_processed['quantity_mt'] = market_cargoes_processed['Quantity_MT']
# Market cargoes don't have freight rate - will use market rates
market_cargoes_processed['freight_rate_usd_per_mt'] = 0  # Will be filled from freight_rates.csv
market_cargoes_processed['commission_percent'] = market_cargoes_processed['Commission_Percent'] / 100.0
market_cargoes_processed['laycan_start_date'] = pd.to_datetime(market_cargoes_processed['Laycan_Start'])
market_cargoes_processed['laycan_end_date'] = pd.to_datetime(market_cargoes_processed['Laycan_End'])
market_cargoes_processed['load_port'] = market_cargoes_processed['Load_Port'].apply(normalize_port_name)
market_cargoes_processed['discharge_port'] = market_cargoes_processed['Discharge_Port'].apply(normalize_port_name)
# Sum load and discharge port costs
market_cargoes_processed['port_costs_usd'] = (
    market_cargoes_processed['Port_Cost_Load_USD'].fillna(0) + 
    market_cargoes_processed['Port_Cost_Discharge_USD'].fillna(0)
)
market_cargoes_processed['load_turn_time_hours'] = market_cargoes_processed['Load_Turn_Time_hr'].fillna(12)
market_cargoes_processed['discharge_turn_time_hours'] = market_cargoes_processed['Discharge_Turn_Time_hr'].fillna(24)
market_cargoes_processed['route'] = market_cargoes_processed['Route']

# Normalize port distances
print("Normalizing port distances...")
port_distances_processed = port_distances.copy()
port_distances_processed['port_from'] = port_distances_processed['PORT_NAME_FROM'].str.upper().str.replace(" ", "_")
port_distances_processed['port_to'] = port_distances_processed['PORT_NAME_TO'].str.upper().str.replace(" ", "_")
port_distances_processed['distance_nautical_miles'] = port_distances_processed['DISTANCE']

# Create distance lookup dictionary for fast access
# Store multiple variations: full name and port-only name (since Port Distances uses port-only format)
distance_lookup = {}
for _, row in port_distances_processed.iterrows():
    port_from = row['port_from']
    port_to = row['port_to']
    distance = row['distance_nautical_miles']
    
    # Store with original format from Port Distances (usually just port name)
    key = (port_from, port_to)
    distance_lookup[key] = distance
    
    # Also create variations for ports that might have country suffix
    # Extract just the port name (before first underscore if exists)
    port_from_base = port_from.split('_')[0] if '_' in port_from else port_from
    port_to_base = port_to.split('_')[0] if '_' in port_to else port_to
    
    # Store port-only version if different (for reverse lookup)
    if port_from_base != port_from or port_to_base != port_to:
        key_base = (port_from_base, port_to_base)
        if key_base not in distance_lookup:  # Don't overwrite if already exists
            distance_lookup[key_base] = distance

print("\n✓ Data preprocessing complete!")
print(f"Processed {len(cargill_vessels_processed)} Cargill vessels")
print(f"Processed {len(market_vessels_processed)} market vessels")
print(f"Processed {len(cargill_cargoes_processed)} Cargill cargoes")
print(f"Processed {len(market_cargoes_processed)} market cargoes")
print(f"Distance lookup entries: {len(distance_lookup)}")

Preprocessing vessels...
Preprocessing cargoes...
Normalizing port distances...

✓ Data preprocessing complete!
Processed 4 Cargill vessels
Processed 11 market vessels
Processed 3 Cargill cargoes
Processed 8 market cargoes
Distance lookup entries: 22153


In [6]:
# Cell 3: Build bunker price lookup function

# Reshape bunker forward curve from wide to long format
bunker_long = []
date_cols = [col for col in bunker_forward_curve.columns if re.match(r'^\d{4}-\d{2}-\d{2}$', col)]

for _, row in bunker_forward_curve.iterrows():
    location = row['location']
    fuel_grade = row['fuel_grade']
    lat = row['latitude']
    lon = row['longitude']
    for date_col in date_cols:
        price = row[date_col]
        if pd.notna(price):
            date = pd.to_datetime(date_col)
            bunker_long.append({
                'location': location,
                'fuel_grade': fuel_grade,
                'latitude': lat,
                'longitude': lon,
                'date': date,
                'price': price
            })

bunker_df = pd.DataFrame(bunker_long)
bunker_df = bunker_df.sort_values(['location', 'fuel_grade', 'date'])

print(f"Bunker data points: {len(bunker_df)}")
print(f"Locations: {bunker_df['location'].nunique()}")
print(f"Date range: {bunker_df['date'].min()} to {bunker_df['date'].max()}")

# Create location mapping from port names to bunker locations
# Use port_locations to match ports to bunker curve locations by lat/lon proximity
port_to_bunker_location = {}
for _, port_row in port_locations.iterrows():
    port_name = port_row['port_name']
    port_lat = port_row['latitude']
    port_lon = port_row['longitude']
    
    # Find nearest bunker location by distance
    min_dist = float('inf')
    nearest_location = None
    for _, bunker_row in bunker_forward_curve.iterrows():
        bunker_lat = bunker_row['latitude']
        bunker_lon = bunker_row['longitude']
        # Simple Euclidean distance (approximate)
        dist = ((port_lat - bunker_lat)**2 + (port_lon - bunker_lon)**2)**0.5
        if dist < min_dist:
            min_dist = dist
            nearest_location = bunker_row['location']
    
    # Extract port name from "Port_Country" format
    port_base = port_name.split('_')[0] if '_' in port_name else port_name
    port_to_bunker_location[port_name] = nearest_location
    # Also map the base port name
    port_to_bunker_location[port_base] = nearest_location

def get_bunker_price(port_name, fuel_grade='VLSFO', date=None):
    """
    Get bunker price for a port and date.
    
    Args:
        port_name: Port name (normalized or original format)
        fuel_grade: 'VLSFO' or 'MGO'
        date: datetime object or date
    
    Returns:
        Price in USD/MT, or None if not found
    """
    if date is None:
        return None
    
    if isinstance(date, datetime):
        date = date.date()
    elif isinstance(date, pd.Timestamp):
        date = date.date()
    
    # Normalize port name
    port_normalized = normalize_port_name(port_name) if port_name else None
    if not port_normalized:
        return None
    
    # Try to find bunker location
    # First try exact match in port_to_bunker_location
    bunker_location = None
    if port_normalized in port_to_bunker_location:
        bunker_location = port_to_bunker_location[port_normalized]
    else:
        # Try to extract base port name
        port_base = port_normalized.split('_')[0] if '_' in port_normalized else port_normalized
        if port_base in port_to_bunker_location:
            bunker_location = port_to_bunker_location[port_base]
        else:
            # Try direct match in bunker_forward_curve
            matching = bunker_forward_curve[bunker_forward_curve['location'].str.upper() == port_base.upper()]
            if len(matching) > 0:
                bunker_location = matching.iloc[0]['location']
    
    if not bunker_location:
        # Fallback: use average price across all locations
        matching = bunker_df[(bunker_df['fuel_grade'] == fuel_grade) & (bunker_df['date'] == pd.Timestamp(date))]
        if len(matching) > 0:
            return matching['price'].mean()
        return None
    
    # Get prices for this location and fuel grade
    location_prices = bunker_df[(bunker_df['location'] == bunker_location) & 
                                 (bunker_df['fuel_grade'] == fuel_grade)].copy()
    
    if len(location_prices) == 0:
        # Try VLSFO if MGO not available
        if fuel_grade == 'MGO':
            location_prices = bunker_df[(bunker_df['location'] == bunker_location) & 
                                         (bunker_df['fuel_grade'] == 'VLSFO')].copy()
        if len(location_prices) == 0:
            return None
    
    # Interpolate between dates
    location_prices = location_prices.sort_values('date')
    target_date = pd.Timestamp(date)
    
    # Exact match
    exact = location_prices[location_prices['date'] == target_date]
    if len(exact) > 0:
        return exact.iloc[0]['price']
    
    # Before first date
    if target_date < location_prices['date'].min():
        return location_prices.iloc[0]['price']
    
    # After last date
    if target_date > location_prices['date'].max():
        return location_prices.iloc[-1]['price']
    
    # Interpolate between two dates
    before = location_prices[location_prices['date'] < target_date].iloc[-1]
    after = location_prices[location_prices['date'] > target_date].iloc[0]
    
    days_between = (after['date'] - before['date']).days
    days_to_target = (target_date - before['date']).days
    
    if days_between == 0:
        return before['price']
    
    weight = days_to_target / days_between
    interpolated_price = before['price'] * (1 - weight) + after['price'] * weight
    
    return interpolated_price

# Test the function
test_date = pd.to_datetime('2026-03-15')
test_price = get_bunker_price('Qingdao_China', 'VLSFO', test_date)
print(f"\nTest: Bunker price for Qingdao on {test_date.date()}: ${test_price:.2f}/MT")
print("✓ Bunker price lookup function ready")

Bunker data points: 216
Locations: 9
Date range: 2026-02-01 00:00:00 to 2027-01-01 00:00:00

Test: Bunker price for Qingdao on 2026-03-15: $641.19/MT
✓ Bunker price lookup function ready


In [7]:
# Cell 4: Build freight rate matching and lookup function

# Reshape freight rates from wide to long format
freight_long = []
period_cols = [col for col in freight_rates.columns if col != 'route']

for _, row in freight_rates.iterrows():
    route = row['route']
    for period_col in period_cols:
        rate = row[period_col]
        if pd.notna(rate) and rate != '':
            freight_long.append({
                'route': route,
                'period': period_col,
                'rate': rate
            })

freight_rates_long = pd.DataFrame(freight_long)

print(f"Freight rate data points: {len(freight_rates_long)}")
print(f"Routes: {freight_rates_long['route'].nunique()}")

# Route matching logic
def match_route_to_freight_rate(cargo_route):
    """
    Match a cargo route (e.g., "Brazil – China") to a freight rate route.
    
    Returns:
        Matched route name from freight_rates.csv, or None
    """
    if pd.isna(cargo_route):
        return None
    
    route_str = str(cargo_route).strip()
    
    # Manual mapping for common routes
    route_mapping = {
        'Brazil – China': 'C3 (Tubarao-Qingdao)',
        'Brazil – China (Iron Ore)': 'C3 (Tubarao-Qingdao)',
        'Australia – China': 'C5 (West Australia-Qingdao)',
        'Australia – China (Iron Ore)': 'C5 (West Australia-Qingdao)',
        'West Africa – China': None,  # No direct match
        'West Africa – India': None,
        'South Africa – China': None,
        'Indonesia – India': None,
        'Canada – China': None,
        'Australia – South Korea': None,
        'Brazil – Malaysia': None,
    }
    
    if route_str in route_mapping:
        return route_mapping[route_str]
    
    # Try pattern matching
    route_upper = route_str.upper()
    for freight_route in freight_rates['route'].unique():
        freight_route_clean = str(freight_route).strip().upper()
        # Check if key terms match
        if 'BRAZIL' in route_upper and 'CHINA' in route_upper and 'C3' in freight_route_clean:
            return freight_route
        if 'AUSTRALIA' in route_upper and 'CHINA' in route_upper and 'C5' in freight_route_clean:
            return freight_route
    
    return None

def get_market_freight_rate(cargo_route, laycan_date):
    """
    Get market freight rate for a cargo route and laycan period.
    
    Args:
        cargo_route: Route string from cargo data
        laycan_date: datetime or date for the laycan
    
    Returns:
        Market freight rate in USD/MT, or None
    """
    matched_route = match_route_to_freight_rate(cargo_route)
    if not matched_route:
        return None
    
    if isinstance(laycan_date, datetime):
        laycan_date = laycan_date.date()
    elif isinstance(laycan_date, pd.Timestamp):
        laycan_date = laycan_date.date()
    
    # Match period
    year = laycan_date.year
    month = laycan_date.month
    quarter = (month - 1) // 3 + 1
    
    # Try to match period columns
    period_candidates = [
        f"{year}-{month:02d}",  # e.g., "2026-03"
        f"{year}-Q{quarter}",   # e.g., "2026-Q1"
        str(year),               # e.g., "2026"
    ]
    
    for period in period_candidates:
        matching = freight_rates_long[
            (freight_rates_long['route'] == matched_route) & 
            (freight_rates_long['period'] == period)
        ]
        if len(matching) > 0:
            return matching.iloc[0]['rate']
    
    # If no exact match, try to find closest period
    route_rates = freight_rates_long[freight_rates_long['route'] == matched_route]
    if len(route_rates) > 0:
        # Return average or first available
        return route_rates['rate'].iloc[0]
    
    return None

# Test the function
test_route = "Brazil – China"
test_laycan = pd.to_datetime('2026-03-15')
test_rate = get_market_freight_rate(test_route, test_laycan)
print(f"\nTest: Market freight rate for '{test_route}' in {test_laycan.date()}: ${test_rate:.2f}/MT")
print("✓ Freight rate lookup function ready")

Freight rate data points: 49
Routes: 4

Test: Market freight rate for 'Brazil – China' in 2026-03-15: $20908.00/MT
✓ Freight rate lookup function ready


In [8]:
# Cell 5: Implement compute_profit function

def compute_profit(vessel_row, cargo_row, distance_lookup, get_bunker_price_fn, get_market_freight_rate_fn):
    """
    Compute profit for a vessel-cargo assignment.
    
    Returns:
        dict with keys: 'profit', 'revenue', 'costs', 'opportunity_cost', 'market_rate', 
        'days_ballast', 'days_laden', 'total_days', or None if infeasible
    """
    try:
        # Feasibility checks
        # 1. Capacity check
        if cargo_row['quantity_mt'] > vessel_row['deadweight_tonnage_dwt']:
            return None
        
        # 2. Distance lookups
        ballast_port_from = vessel_row['current_position_port']
        ballast_port_to = cargo_row['load_port']
        laden_port_from = cargo_row['load_port']
        laden_port_to = cargo_row['discharge_port']
        
        if not ballast_port_from or not ballast_port_to or not laden_port_to:
            return None
        
        # Helper function to get distance with multiple port name format attempts
        def get_distance(port_from, port_to):
            """Try multiple port name formats to find distance"""
            if not port_from or not port_to:
                return None
            
            # Try exact match first (e.g., "QINGDAO_CHINA", "KAMSAR_GUINEA")
            distance = distance_lookup.get((port_from, port_to))
            if distance is not None:
                return distance
            
            # Extract port name only (before underscore) - Port Distances uses this format
            port_from_base = port_from.split('_')[0] if port_from and '_' in port_from else port_from
            port_to_base = port_to.split('_')[0] if port_to and '_' in port_to else port_to
            
            # Try port-only match (e.g., "QINGDAO", "KAMSAR")
            distance = distance_lookup.get((port_from_base, port_to_base))
            if distance is not None:
                return distance
            
            # Try reverse direction (sometimes distances are stored in reverse)
            distance = distance_lookup.get((port_to_base, port_from_base))
            if distance is not None:
                return distance
            
            # Try with one port full name and one port-only
            distance = distance_lookup.get((port_from, port_to_base))
            if distance is not None:
                return distance
            
            distance = distance_lookup.get((port_from_base, port_to))
            if distance is not None:
                return distance
            
            return None
        
        ballast_distance = get_distance(ballast_port_from, ballast_port_to)
        laden_distance = get_distance(laden_port_from, laden_port_to)
        
        if ballast_distance is None or laden_distance is None:
            return None
        
        # 3. Time calculations
        # Ballast leg
        ballast_speed = vessel_row['economical_speed_knots']
        days_ballast = ballast_distance / (ballast_speed * 24.0)
        
        # Laden leg
        laden_speed = vessel_row['speed_laden']
        days_laden = laden_distance / (laden_speed * 24.0)
        
        # Port time
        port_days = (cargo_row['load_turn_time_hours'] + cargo_row['discharge_turn_time_hours']) / 24.0
        
        total_days = days_ballast + days_laden + port_days
        
        # 4. Laycan check
        etd = vessel_row['estimated_time_of_departure']
        if isinstance(etd, str):
            etd = pd.to_datetime(etd)
        arrival_at_load = etd + timedelta(days=days_ballast)
        
        if arrival_at_load.date() > cargo_row['laycan_end_date'].date():
            return None
        
        # 5. Revenue
        freight_rate = cargo_row['freight_rate_usd_per_mt']
        quantity = cargo_row['quantity_mt']
        revenue = freight_rate * quantity
        
        # 6. Costs
        # Commission
        commission = cargo_row['commission_percent'] * revenue
        
        # Fuel costs
        # Ballast fuel
        fuel_ballast = vessel_row['sea_consumption_mt_per_day'] * days_ballast
        bunker_price_ballast = get_bunker_price_fn(ballast_port_from, 'VLSFO', etd.date())
        if bunker_price_ballast is None:
            bunker_price_ballast = 500  # Default fallback
        
        # Laden fuel
        fuel_laden = vessel_row['consumption_laden'] * days_laden
        bunker_price_laden = get_bunker_price_fn(laden_port_from, 'VLSFO', arrival_at_load.date())
        if bunker_price_laden is None:
            bunker_price_laden = 500  # Default fallback
        
        # Port fuel
        fuel_port = vessel_row['port_consumption_mt_per_day'] * port_days
        bunker_price_port = get_bunker_price_fn(laden_port_from, 'VLSFO', arrival_at_load.date())
        if bunker_price_port is None:
            bunker_price_port = 500  # Default fallback
        
        fuel_cost = (fuel_ballast * bunker_price_ballast + 
                    fuel_laden * bunker_price_laden + 
                    fuel_port * bunker_price_port)
        
        # Hire cost
        hire_cost = vessel_row['hire_rate_usd_per_day'] * total_days
        
        # Port costs
        port_costs = cargo_row['port_costs_usd']
        
        total_costs = commission + fuel_cost + hire_cost + port_costs
        
        # Profit
        profit = revenue - total_costs
        
        # 7. Opportunity cost (market rate comparison)
        market_rate = get_market_freight_rate_fn(cargo_row.get('route'), cargo_row['laycan_start_date'])
        opportunity_cost = None
        if market_rate is not None:
            market_revenue = market_rate * quantity
            opportunity_cost = market_revenue - revenue  # Positive if market rate is higher
        
        return {
            'profit': profit,
            'revenue': revenue,
            'costs': total_costs,
            'commission': commission,
            'fuel_cost': fuel_cost,
            'hire_cost': hire_cost,
            'port_costs': port_costs,
            'opportunity_cost': opportunity_cost,
            'market_rate': market_rate,
            'offered_rate': freight_rate,
            'days_ballast': days_ballast,
            'days_laden': days_laden,
            'total_days': total_days,
            'ballast_distance': ballast_distance,
            'laden_distance': laden_distance
        }
    
    except Exception as e:
        print(f"Error computing profit: {e}")
        return None

print("✓ compute_profit function implemented")

✓ compute_profit function implemented


In [9]:
# Install ortools
%pip install ortools

  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached immutabledict-4.2.2-py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 9.4 MB/s  0:00:02 eta 0:00:01
Using cached absl_py-2.3.1-py3-none-any.whl (135 kB)
Using cached immutabledict-4.2.2-py3-none-any.whl (4.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ortools]m3/4 [ortools]edict]
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Cell 6: Implement optimize_assignments using OR-Tools CP-SAT

from ortools.sat.python import cp_model

def optimize_assignments(cargill_vessels_df, market_vessels_df, 
                         cargill_cargoes_df, market_cargoes_df,
                         distance_lookup, get_bunker_price_fn, get_market_freight_rate_fn):
    """
    Optimize vessel-cargo assignments using CP-SAT.
    
    Returns:
        assignments_df: DataFrame with optimal assignments
        total_profit: Total profit from all assignments
    """
    model = cp_model.CpModel()
    
    # Build all feasible pairs and compute profits
    feasible_pairs = []
    
    # Cargill vessels -> Cargill cargoes
    for v_idx, vessel in cargill_vessels_df.iterrows():
        for c_idx, cargo in cargill_cargoes_df.iterrows():
            profit_data = compute_profit(vessel, cargo, distance_lookup, 
                                         get_bunker_price_fn, get_market_freight_rate_fn)
            if profit_data is not None:
                feasible_pairs.append({
                    'origin': 'cargill',
                    'vessel_idx': v_idx,
                    'cargo_idx': c_idx,
                    'cargo_type': 'committed',
                    'profit_data': profit_data
                })
    
    # Cargill vessels -> Market cargoes
    for v_idx, vessel in cargill_vessels_df.iterrows():
        for c_idx, cargo in market_cargoes_df.iterrows():
            profit_data = compute_profit(vessel, cargo, distance_lookup,
                                         get_bunker_price_fn, get_market_freight_rate_fn)
            if profit_data is not None:
                feasible_pairs.append({
                    'origin': 'cargill',
                    'vessel_idx': v_idx,
                    'cargo_idx': c_idx,
                    'cargo_type': 'market',
                    'profit_data': profit_data
                })
    
    # Market vessels -> Cargill cargoes
    for v_idx, vessel in market_vessels_df.iterrows():
        for c_idx, cargo in cargill_cargoes_df.iterrows():
            profit_data = compute_profit(vessel, cargo, distance_lookup,
                                         get_bunker_price_fn, get_market_freight_rate_fn)
            if profit_data is not None:
                feasible_pairs.append({
                    'origin': 'market',
                    'vessel_idx': v_idx,
                    'cargo_idx': c_idx,
                    'cargo_type': 'committed',
                    'profit_data': profit_data
                })
    
    print(f"Found {len(feasible_pairs)} feasible vessel-cargo pairs")
    
    if len(feasible_pairs) == 0:
        print("No feasible pairs found!")
        return pd.DataFrame(), 0
    
    # Create decision variables
    variables = {}
    for i, pair in enumerate(feasible_pairs):
        var_name = f"x_{pair['origin']}_{pair['vessel_idx']}_{pair['cargo_type']}_{pair['cargo_idx']}"
        variables[i] = model.NewBoolVar(var_name)
    
    # Constraints
    # 1. Each committed cargo must be assigned exactly once
    committed_cargo_indices = set()
    for i, pair in enumerate(feasible_pairs):
        if pair['cargo_type'] == 'committed':
            committed_cargo_indices.add((pair['origin'], pair['cargo_idx']))
    
    for (origin, cargo_idx) in committed_cargo_indices:
        # Find all pairs for this committed cargo
        pairs_for_cargo = [i for i, p in enumerate(feasible_pairs) 
                          if p['cargo_type'] == 'committed' and p['cargo_idx'] == cargo_idx]
        if len(pairs_for_cargo) > 0:
            model.Add(sum(variables[i] for i in pairs_for_cargo) == 1)
    
    # 2. Each market cargo can be assigned at most once
    market_cargo_indices = set()
    for i, pair in enumerate(feasible_pairs):
        if pair['cargo_type'] == 'market':
            market_cargo_indices.add((pair['origin'], pair['cargo_idx']))
    
    for (origin, cargo_idx) in market_cargo_indices:
        pairs_for_cargo = [i for i, p in enumerate(feasible_pairs) 
                          if p['cargo_type'] == 'market' and p['cargo_idx'] == cargo_idx]
        if len(pairs_for_cargo) > 0:
            model.Add(sum(variables[i] for i in pairs_for_cargo) <= 1)
    
    # 3. Each vessel can be assigned at most once
    vessel_indices = set()
    for i, pair in enumerate(feasible_pairs):
        vessel_indices.add((pair['origin'], pair['vessel_idx']))
    
    for (origin, vessel_idx) in vessel_indices:
        pairs_for_vessel = [i for i, p in enumerate(feasible_pairs) 
                           if p['origin'] == origin and p['vessel_idx'] == vessel_idx]
        if len(pairs_for_vessel) > 0:
            model.Add(sum(variables[i] for i in pairs_for_vessel) <= 1)
    
    # Objective: Maximize total profit
    # Scale profit to integers (multiply by 100)
    objective_terms = []
    for i, pair in enumerate(feasible_pairs):
        profit_scaled = int(round(pair['profit_data']['profit'] * 100))
        objective_terms.append(variables[i] * profit_scaled)
    
    model.Maximize(sum(objective_terms))
    
    # Solve
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 300.0  # 5 minutes
    status = solver.Solve(model)
    
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print(f"Solution status: {'OPTIMAL' if status == cp_model.OPTIMAL else 'FEASIBLE'}")
        
        # Extract assignments
        assignments = []
        total_profit = 0
        
        for i, pair in enumerate(feasible_pairs):
            if solver.Value(variables[i]) == 1:
                profit_data = pair['profit_data']
                total_profit += profit_data['profit']
                
                # Get vessel and cargo details
                if pair['origin'] == 'cargill':
                    vessel = cargill_vessels_df.loc[pair['vessel_idx']]
                    vessel_name = vessel['vessel_name']
                else:
                    vessel = market_vessels_df.loc[pair['vessel_idx']]
                    vessel_name = vessel['vessel_name']
                
                if pair['cargo_type'] == 'committed':
                    cargo = cargill_cargoes_df.loc[pair['cargo_idx']]
                    cargo_id = cargo['cargo_id']
                    load_port = cargo['load_port']
                    discharge_port = cargo['discharge_port']
                else:
                    cargo = market_cargoes_df.loc[pair['cargo_idx']]
                    cargo_id = cargo['cargo_id']
                    load_port = cargo['load_port']
                    discharge_port = cargo['discharge_port']
                
                assignments.append({
                    'Vessel_Type': 'Cargill' if pair['origin'] == 'cargill' else 'Market',
                    'Vessel_Name': vessel_name,
                    'Cargo_Type': 'Committed' if pair['cargo_type'] == 'committed' else 'Market',
                    'Cargo_ID': cargo_id,
                    'Load_Port': load_port,
                    'Discharge_Port': discharge_port,
                    'Profit': profit_data['profit'],
                    'Revenue': profit_data['revenue'],
                    'Costs': profit_data['costs'],
                    'TCE': profit_data['profit'] / profit_data['total_days'] if profit_data['total_days'] > 0 else 0,
                    'Voyage_Days': profit_data['total_days'],
                    'Days_Ballast': profit_data['days_ballast'],
                    'Days_Laden': profit_data['days_laden'],
                    'Offered_Rate': profit_data['offered_rate'],
                    'Market_Rate': profit_data['market_rate'],
                    'Opportunity_Cost': profit_data['opportunity_cost'],
                    'Ballast_Distance': profit_data['ballast_distance'],
                    'Laden_Distance': profit_data['laden_distance']
                })
        
        assignments_df = pd.DataFrame(assignments)
        return assignments_df, total_profit
    
    else:
        print(f"Solution status: {solver.StatusName(status)}")
        return pd.DataFrame(), 0

print("✓ optimize_assignments function implemented")

✓ optimize_assignments function implemented


In [11]:
# Cell 7: Run the optimizer

print("Running optimization...")
print("=" * 60)

assignments_df, total_profit = optimize_assignments(
    cargill_vessels_processed,
    market_vessels_processed,
    cargill_cargoes_processed,
    market_cargoes_processed,
    distance_lookup,
    get_bunker_price,
    get_market_freight_rate
)

print("\n" + "=" * 60)
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Number of Assignments: {len(assignments_df)}")
print("=" * 60)

if len(assignments_df) > 0:
    print("\nAssignments (sorted by profit):")
    print(assignments_df.sort_values('Profit', ascending=False).to_string(index=False))
    
    print("\n\nSummary Statistics:")
    print(f"Average Profit: ${assignments_df['Profit'].mean():,.2f}")
    print(f"Average TCE: ${assignments_df['TCE'].mean():,.2f}")
    print(f"Total Voyage Days: {assignments_df['Voyage_Days'].sum():.1f}")
    
    if assignments_df['Market_Rate'].notna().any():
        print("\nMarket Rate Comparisons:")
        market_comparisons = assignments_df[assignments_df['Market_Rate'].notna()].copy()
        if len(market_comparisons) > 0:
            print(f"  Cargoes with market rate data: {len(market_comparisons)}")
            print(f"  Average opportunity cost: ${market_comparisons['Opportunity_Cost'].mean():,.2f}")
else:
    print("\nNo assignments found. Check constraints and data.")

Running optimization...
Found 10 feasible vessel-cargo pairs
Solution status: OPTIMAL

Total Profit: $1,923,817.81
Number of Assignments: 2

Assignments (sorted by profit):
Vessel_Type   Vessel_Name Cargo_Type  Cargo_ID              Load_Port    Discharge_Port       Profit   Revenue        Costs          TCE  Voyage_Days  Days_Ballast  Days_Laden  Offered_Rate  Market_Rate  Opportunity_Cost  Ballast_Distance  Laden_Distance
    Cargill      ANN BELL  Committed CARGILL_1          KAMSAR_GUINEA     QINGDAO_CHINA 1.467322e+06 4140000.0 2.672678e+06 19129.418850    76.705000     37.080000   38.625000          23.0          NaN               NaN          11124.00         11124.0
     Market CORAL EMPEROR  Committed CARGILL_2 PORT_HEDLAND_AUSTRALIA LIANYUNGANG_CHINA 4.564957e+05 1440000.0 9.835043e+05 15742.333024    28.997972      0.939499   26.558473           9.0       8717.0      1393280000.0            277.34          7585.1


Summary Statistics:
Average Profit: $961,908.90
Average TCE:

In [12]:
# Optional: Visualize optimal routes on a map

if len(assignments_df) > 0:
    import folium
    
    # Create base map
    m = folium.Map(location=[20, 0], zoom_start=2)
    
    # Color mapping
    colors = {
        'Cargill': 'blue',
        'Market': 'green'
    }
    
    # Add routes
    for _, assignment in assignments_df.iterrows():
        # Get port coordinates
        load_port_name = assignment['Load_Port']
        discharge_port_name = assignment['Discharge_Port']
        
        # Find coordinates from port_locations
        load_coords = None
        disc_coords = None
        
        for _, port_row in port_locations.iterrows():
            port_normalized = normalize_port_name(port_row['port_name'])
            if port_normalized == load_port_name:
                load_coords = [port_row['latitude'], port_row['longitude']]
            if port_normalized == discharge_port_name:
                disc_coords = [port_row['latitude'], port_row['longitude']]
        
        if load_coords and disc_coords:
            # Add line
            folium.PolyLine(
                [load_coords, disc_coords],
                color=colors.get(assignment['Vessel_Type'], 'gray'),
                weight=3,
                opacity=0.7,
                popup=f"{assignment['Vessel_Name']} → {assignment['Cargo_ID']}<br>Profit: ${assignment['Profit']:,.0f}"
            ).add_to(m)
            
            # Add markers
            folium.CircleMarker(
                load_coords,
                radius=8,
                popup=f"Load: {load_port_name}",
                color=colors.get(assignment['Vessel_Type'], 'gray'),
                fill=True
            ).add_to(m)
            
            folium.CircleMarker(
                disc_coords,
                radius=8,
                popup=f"Discharge: {discharge_port_name}",
                color=colors.get(assignment['Vessel_Type'], 'gray'),
                fill=True
            ).add_to(m)
    
    # Save map
    m.save('optimal_routes_map.html')
    print("Map saved to optimal_routes_map.html")
else:
    print("No assignments to visualize")

Map saved to optimal_routes_map.html
